# Test Gold DW — Validacion completa del star schema

Verifica el Data Warehouse Gold (`gold_ss2`) en 4 niveles:

| Nivel | Que verifica |
|---|---|
| 1. Existencia | Tablas y columnas presentes, row counts > 0 |
| 2. Dimensiones | Valores esperados en cada dim (placeholders, rangos, unicidad) |
| 3. Integridad FK | NULLs en claves foraneas de las facts |
| 4. Queries analiticas | Joins end-to-end con resultados coherentes |

In [0]:
from pyspark.sql import functions as F

GOLD = 'gold_ss2'

_DIMS  = ['dim_tiempo', 'dim_geografia', 'dim_causa', 'dim_demografia', 'dim_fuente']
_FACTS = ['fact_morbilidad', 'fact_defunciones']
_ALL   = _DIMS + _FACTS

results = []  # (check_name, passed, detail)

def check(name, passed, detail=''):
    results.append((name, passed, detail))
    icon = 'PASS' if passed else 'FAIL'
    print(f'  [{icon}] {name}' + (f' — {detail}' if detail else ''))

print('Setup OK — Gold schema:', GOLD)

---
## Nivel 1 — Existencia de tablas y row counts

In [0]:
print(f'{'Tabla':<35} {'Filas':>12} {'Estado':>6}')
print('-' * 56)

counts = {}
for t in _ALL:
    try:
        n = spark.table(f'{GOLD}.{t}').count()
        counts[t] = n
        passed = n > 0
        check(f'existencia {t}', passed, f'{n:,} filas')
        print(f'  {t:<33} {n:>12,}')
    except Exception as e:
        counts[t] = 0
        check(f'existencia {t}', False, str(e))
        print(f'  {t:<33} ERROR: {e}')

---
## Nivel 2 — Sanity checks por dimension

### dim_tiempo

In [0]:
df_t = spark.table(f'{GOLD}.dim_tiempo')
display(df_t.orderBy('anio'))

# Debe tener exactamente 14 filas (2012-2025)
check('dim_tiempo: 14 filas', df_t.count() == 14, f'{df_t.count()} filas')

# Rango de anos correcto
rng = df_t.agg(F.min('anio').alias('mn'), F.max('anio').alias('mx')).first()
check('dim_tiempo: rango 2012-2025', rng['mn'] == 2012 and rng['mx'] == 2025, f'{rng["mn"]}-{rng["mx"]}')

# Periodos COVID correctos
post = df_t.filter(F.col('periodo_covid') == 'covid_y_post').agg(F.min('anio'), F.max('anio')).first()
check('dim_tiempo: covid_y_post empieza en 2020', post[0] == 2020, f'desde {post[0]}')

# No hay duplicados en anio
dupes = df_t.count() - df_t.dropDuplicates(['anio']).count()
check('dim_tiempo: sin duplicados en anio', dupes == 0, f'{dupes} duplicados')

### dim_geografia

In [0]:
df_g = spark.table(f'{GOLD}.dim_geografia')

print('Distribucion por nivel_geo:')
display(df_g.groupBy('nivel_geo').count().orderBy('nivel_geo'))

print('Filas nivel pais:')
display(df_g.filter(F.col('nivel_geo') == 'pais'))

# Placeholder INE-edad debe existir
placeholder = df_g.filter(
    (F.col('pais') == 'Guatemala') & (F.col('departamento') == 'Sin desagregar')
).count()
check('dim_geografia: placeholder INE-edad existe', placeholder == 1, f'{placeholder} fila')

# Debe haber filas de pais para Guatemala y Costa Rica
gt = df_g.filter((F.col('nivel_geo') == 'pais') & (F.col('pais') == 'Guatemala') & F.col('departamento').isNull()).count()
cr = df_g.filter((F.col('nivel_geo') == 'pais') & (F.col('pais') == 'Costa Rica')).count()
check('dim_geografia: pais Guatemala (WHO)', gt == 1, f'{gt} fila')
check('dim_geografia: pais Costa Rica (WHO)', cr == 1, f'{cr} fila')

# Debe haber 22 departamentos
deptos = df_g.filter(F.col('nivel_geo') == 'departamento').count()
check('dim_geografia: 22 departamentos', deptos == 22, f'{deptos} deptos')

# Sin duplicados en surrogate key
dupes = df_g.count() - df_g.dropDuplicates(['geografia_sk']).count()
check('dim_geografia: geografia_sk unico', dupes == 0, f'{dupes} duplicados')

### dim_causa

In [0]:
df_c = spark.table(f'{GOLD}.dim_causa')

print(f'Total causas unicas: {df_c.count():,}')
print('Muestra:')
display(df_c.limit(15))

# No debe haber codigo_origen NULL
null_cod = df_c.filter(F.col('codigo_origen').isNull()).count()
check('dim_causa: sin codigo_origen NULL', null_cod == 0, f'{null_cod} NULLs')

# No debe haber duplicados en codigo_origen (pueden existir si mismo codigo tiene distintas descripciones)
dupes_sk = df_c.count() - df_c.dropDuplicates(['causa_sk']).count()
check('dim_causa: causa_sk unico', dupes_sk == 0, f'{dupes_sk} duplicados')

### dim_demografia

In [0]:
df_d = spark.table(f'{GOLD}.dim_demografia')

print(f'Total combinaciones: {df_d.count()}')
print('Distribucion por sexo:')
display(df_d.groupBy('sexo').count())

# Placeholder Sin desagregar debe existir
ph = df_d.filter(
    (F.col('sexo') == 'Ambos') & (F.col('grupo_etario_label') == 'Sin desagregar')
).count()
check('dim_demografia: placeholder Sin desagregar existe', ph == 1, f'{ph} fila')

# Sexos validos
sexos_invalidos = df_d.filter(~F.col('sexo').isin(['M', 'F', 'Ambos'])).count()
check('dim_demografia: sexos validos (M/F/Ambos)', sexos_invalidos == 0, f'{sexos_invalidos} invalidos')

# Sin duplicados en surrogate key
dupes = df_d.count() - df_d.dropDuplicates(['demografia_sk']).count()
check('dim_demografia: demografia_sk unico', dupes == 0, f'{dupes} duplicados')

print('Muestra ordenada:')
display(df_d.orderBy('sexo', 'grupo_etario_anios_min').limit(20))

### dim_fuente

In [0]:
df_f = spark.table(f'{GOLD}.dim_fuente')

print(f'Total fuentes: {df_f.count()}')
display(df_f.orderBy('pipeline_name', 'source_file'))

# Los 4 pipelines deben estar presentes
pipelines_esperados = {'stage_ine_edad', 'stage_ine_geografia', 'stage_mspas', 'stage_who'}
pipelines_reales = set(r['pipeline_name'] for r in df_f.select('pipeline_name').distinct().collect())
check('dim_fuente: 4 pipelines presentes', pipelines_esperados == pipelines_reales, str(pipelines_reales))

# Sin NULLs en columnas clave
for col in ['source_system', 'source_file', 'pipeline_name', 'nivel_agregacion']:
    nulls = df_f.filter(F.col(col).isNull()).count()
    check(f'dim_fuente: {col} sin NULLs', nulls == 0, f'{nulls} NULLs')

---
## Nivel 3 — Integridad referencial de las facts (NULLs en FKs)

In [0]:
_FKS = ['tiempo_sk', 'geografia_sk', 'causa_sk', 'demografia_sk', 'fuente_sk']

# NULLs tolerados en geografia_sk para ambas facts:
#   fact_morbilidad  → municipios MSPAS sin match en catalogo de departamentos
#   fact_defunciones → filas INE donde departamento_oficial es NULL en Silver (depto no encontrado en catalogo)
_TOLERADOS = {
    ('fact_morbilidad',  'geografia_sk'),
    ('fact_defunciones', 'geografia_sk'),
}

for fact in _FACTS:
    df_fact = spark.table(f'{GOLD}.{fact}')
    total = df_fact.count()
    print(f'\n-- {fact} ({total:,} filas) --')
    for fk in _FKS:
        nulls = df_fact.filter(F.col(fk).isNull()).count()
        pct = 100 * nulls / total if total > 0 else 0
        tolerado = (fact, fk) in _TOLERADOS
        passed = nulls == 0 or tolerado
        nota = ' (tolerado — sin match en catalogo)' if tolerado and nulls > 0 else ''
        check(f'{fact}.{fk} integridad', passed, f'{nulls:,} NULLs ({pct:.1f}%){nota}')

---
## Nivel 4 — Queries analiticas end-to-end

### Q1 — Defunciones totales por ano (INE-geo, nivel departamento)

In [0]:
df_q1 = (
    spark.table(f'{GOLD}.fact_defunciones')
    .join(spark.table(f'{GOLD}.dim_tiempo').select('tiempo_sk', 'anio'), 'tiempo_sk')
    .join(spark.table(f'{GOLD}.dim_fuente').select('fuente_sk', 'pipeline_name'), 'fuente_sk')
    .filter(F.col('pipeline_name') == 'stage_ine_geografia')
    .groupBy('anio')
    .agg(F.sum('total_defunciones').alias('defunciones_totales'))
    .orderBy('anio')
)
display(df_q1)

filas_q1 = df_q1.count()
check('Q1 defunciones INE-geo por anio: tiene filas', filas_q1 > 0, f'{filas_q1} anos')

### Q2 — Top 10 causas de morbilidad (MSPAS, acumulado)

In [0]:
df_q2 = (
    spark.table(f'{GOLD}.fact_morbilidad')
    .join(spark.table(f'{GOLD}.dim_causa').select('causa_sk', 'descripcion'), 'causa_sk')
    .groupBy('descripcion')
    .agg(F.sum('casos_total').alias('total_casos'))
    .orderBy(F.desc('total_casos'))
    .limit(10)
)
display(df_q2)

check('Q2 top causas morbilidad: tiene filas', df_q2.count() > 0)

### Q3 — Defunciones WHO por pais y periodo COVID

In [0]:
df_q3 = (
    spark.table(f'{GOLD}.fact_defunciones')
    .join(spark.table(f'{GOLD}.dim_tiempo').select('tiempo_sk', 'anio', 'periodo_covid'), 'tiempo_sk')
    .join(spark.table(f'{GOLD}.dim_geografia').select('geografia_sk', 'pais', 'nivel_geo'), 'geografia_sk')
    .join(spark.table(f'{GOLD}.dim_fuente').select('fuente_sk', 'pipeline_name'), 'fuente_sk')
    .filter(F.col('pipeline_name') == 'stage_who')
    .filter(F.col('nivel_geo') == 'pais')
    .groupBy('pais', 'periodo_covid')
    .agg(
        F.sum('total_defunciones').alias('defunciones'),
        F.avg('tasa_mortalidad_x100k').alias('tasa_mortalidad_promedio')
    )
    .orderBy('pais', 'periodo_covid')
)
display(df_q3)

check('Q3 WHO por pais y periodo: tiene filas', df_q3.count() > 0)

### Q4 — Morbilidad MSPAS por departamento (pre vs post COVID)

In [0]:
df_q4 = (
    spark.table(f'{GOLD}.fact_morbilidad')
    .join(spark.table(f'{GOLD}.dim_tiempo').select('tiempo_sk', 'periodo_covid'), 'tiempo_sk')
    .join(spark.table(f'{GOLD}.dim_geografia').select('geografia_sk', 'departamento', 'nivel_geo'), 'geografia_sk')
    .filter(F.col('nivel_geo') == 'municipio')  # MSPAS es nivel municipio
    .filter(F.col('departamento').isNotNull())
    .groupBy('departamento', 'periodo_covid')
    .agg(F.sum('casos_total').alias('casos'))
    .groupBy('departamento')
    .pivot('periodo_covid', ['pre_covid', 'covid_y_post'])
    .agg(F.sum('casos'))
    .orderBy(F.desc('covid_y_post'))
    .limit(22)
)
display(df_q4)

check('Q4 morbilidad por depto pre/post COVID: tiene filas', df_q4.count() > 0)

### Q5 — Defunciones INE-edad por grupo etario (Guatemala, sin depto)

In [0]:
df_q5 = (
    spark.table(f'{GOLD}.fact_defunciones')
    .join(spark.table(f'{GOLD}.dim_fuente').select('fuente_sk', 'pipeline_name'), 'fuente_sk')
    .filter(F.col('pipeline_name') == 'stage_ine_edad')
    .join(spark.table(f'{GOLD}.dim_demografia').select('demografia_sk', 'grupo_etario_label', 'grupo_etario_anios_min'), 'demografia_sk')
    .groupBy('grupo_etario_label', 'grupo_etario_anios_min')
    .agg(F.sum('total_defunciones').alias('defunciones'))
    .orderBy('grupo_etario_anios_min')
)
display(df_q5)

check('Q5 defunciones por grupo etario INE: tiene filas', df_q5.count() > 0)

---
## Resumen final

In [0]:
passed  = [r for r in results if r[1]]
failed  = [r for r in results if not r[1]]

print(f'{'='*65}')
print(f'RESUMEN TEST GOLD DW')
print(f'{'='*65}')
print(f'Total checks : {len(results)}')
print(f'Pasados      : {len(passed)}')
print(f'Fallidos     : {len(failed)}')
print()

if failed:
    print('CHECKS FALLIDOS:')
    for name, _, detail in failed:
        print(f'  [FAIL] {name}' + (f' — {detail}' if detail else ''))
else:
    print('Todos los checks pasaron. El DW Gold esta listo.')